# 05 — Analiza wyników DeepEval

Czyta `output/deepeval/deepeval_*.csv` (batch `eval-main`) i buduje **macierz średnich**, **radar** oraz **heatmapy** (typ pytania / kategoria arXiv × warianty **W1–W5**).

**Nazewnictwo metryk jak w DeepEval:**

| Rola | Metryka |
|------|---------|
| Retriever | Contextual Precision, Contextual Recall |
| Generator | Answer Relevancy, Faithfulness |
| (GEval) | Answer Correctness |

**Ablacje i istotność statystyczna** → notebook `08_ablacje_istotnosc.ipynb`.  
**Latency / zasoby** → `07_latency_analysis.ipynb`.

```powershell
cd evaluation
jupyter notebook 05_deepeval_analysis.ipynb
```


In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from scipy import stats

EVAL = Path.cwd() if (Path.cwd() / "output").exists() else Path.cwd() / "evaluation"
load_dotenv(EVAL / ".env")
load_dotenv(EVAL / ".env.example")

BATCH_ID = os.getenv("EVAL_BATCH_ID", "eval-main")
DE = EVAL / "output" / "deepeval"
OFF = EVAL / "output" / "analysis"
FIG = DE / "figures"
FIG.mkdir(parents=True, exist_ok=True)

VARIANT_ORDER = [
    "baseline0", "baseline1", "eksperyment1-gin", "eksperyment1-bm25", "eksperyment2"
]
LABELS = {
    "baseline0": "W1",
    "baseline1": "W2",
    "eksperyment1-gin": "W3",
    "eksperyment1-bm25": "W4",
    "eksperyment2": "W5",
}
METRICS = [
    "faithfulness",
    "answer_relevancy",
    "contextual_recall",
    "contextual_precision",
    "answer_correctness",
]
METRIC_COLS_PL = {
    # Generator
    "faithfulness": "Faithfulness",
    "answer_relevancy": "Answer Relevancy",
    # Retriever
    "contextual_recall": "Contextual Recall",
    "contextual_precision": "Contextual Precision",
    # GEval
    "answer_correctness": "Answer Correctness",
}
QUESTION_TYPE_LABELS_PL = {
    "definicyjne": "Definicyjne",
    "metodologiczne": "Metodologiczne",
    "porownawcze": "Porównawcze",
    "wyniki": "Wyniki",
}
HEATMAP_METRICS = ["answer_correctness", "faithfulness"]

plt.style.use("seaborn-v0_8-whitegrid")

per_q = pd.read_csv(DE / f"deepeval_per_question_{BATCH_ID}.csv")
summary = pd.read_csv(DE / f"deepeval_summary_{BATCH_ID}.csv")

per_q["label"] = per_q["variant"].map(LABELS)
per_q["variant"] = pd.Categorical(per_q["variant"], categories=VARIANT_ORDER, ordered=True)

print("per_question:", len(per_q), "| errors (sędzia DeepEval):", int(per_q["deepeval_error"].notna().sum()))
print("warianty:", {LABELS.get(k, k): int(v) for k, v in per_q["variant"].value_counts().sort_index().items()})



## 1. Macierz DeepEval × wariant (W1–W5)

Wartości w tabeli = **średnie arytmetyczne** (`.mean()`) po pytaniach wariantu — **nie mediany**.  
Na brakujących score’ach (błąd sędziego) `mean` pomija NaN. Kolumna **errors** = pytania z ≥1 błędem DeepEval (timeout / zły JSON) — nie błędy API RAG.

Poniżej radar tylko z 5 metryk DeepEval (bez Hit@5 / MRR).


In [ ]:
rows = []
for v in VARIANT_ORDER:
    g = per_q[per_q["variant"] == v]
    if g.empty:
        continue
    row = {
        "variant": v,
        "label": LABELS[v],
        "n": len(g),
        "errors": int(g["deepeval_error"].notna().sum()),
        "hit_at_5": float(pd.to_numeric(g["hit_at_5"], errors="coerce").mean()),
        "mrr": float(pd.to_numeric(g["mrr"], errors="coerce").mean()) if "mrr" in g else np.nan,
        "idk_rate": float(g["idk"].map(lambda x: bool(x) if pd.notna(x) else False).mean()),
    }
    for m in METRICS:
        row[m] = float(pd.to_numeric(g[m], errors="coerce").mean())
        row[f"{m}_n"] = int(pd.to_numeric(g[m], errors="coerce").notna().sum())
    rows.append(row)

table = pd.DataFrame(rows)
table.to_csv(DE / f"analysis_summary_{BATCH_ID}.csv", index=False)

deepeval_show = table[["label", "n", "errors"] + METRICS].rename(
    columns={"label": "Wariant", "n": "n", "errors": "errors", **METRIC_COLS_PL}
)
display(deepeval_show.round(3))

# Radar: wyłącznie metryki DeepEval (średnie z `table`)
RADAR_COLORS = {
    "baseline0": "#4C72B0",
    "baseline1": "#DD8452",
    "eksperyment1-gin": "#8172B3",
    "eksperyment1-bm25": "#55A868",
    "eksperyment2": "#C44E52",
}
RADAR_LEGEND = {
    "baseline0": "W1: Naive RAG",
    "baseline1": "W2: Parent-Child Chunking",
    "eksperyment1-gin": "W3: Hybrid RAG (GIN)",
    "eksperyment1-bm25": "W4: Hybrid RAG (BM25)",
    "eksperyment2": "W5: W4 + Self-RAG",
}
radar_labels = [METRIC_COLS_PL[m] for m in METRICS]
n_ax = len(radar_labels)
angles = np.linspace(0, 2 * np.pi, n_ax, endpoint=False).tolist()
angles += angles[:1]

fig = plt.figure(figsize=(10.2, 8.2), facecolor="white")
# Radar wyśrodkowany na planszy; miejsce po prawej na legendę
ax = fig.add_axes([0.08, 0.08, 0.62, 0.82], polar=True)
ax.set_facecolor("white")
for _, r in table.iterrows():
    vals = [float(r[m]) for m in METRICS] + [float(r[METRICS[0]])]
    color = RADAR_COLORS.get(r["variant"], "#333333")
    ax.plot(angles, vals, color=color, linewidth=2.4, label=RADAR_LEGEND[r["variant"]], zorder=3)
    ax.fill(angles, vals, color=color, alpha=0.012, zorder=1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=10)
ax.set_ylim(0.0, 1.0)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=9, color="#666666")
ax.grid(color="#CCCCCC", linestyle="--", linewidth=0.8)
ax.spines["polar"].set_color("#B0B0B0")
ax.set_title(
    "Zestawienie metryk DeepEval dla badanych wariantów",
    fontsize=13,
    pad=24,
    color="#333333",
)

ax_leg = fig.add_axes([0.68, 0.55, 0.30, 0.35])
ax_leg.axis("off")
handles, labels = ax.get_legend_handles_labels()
ax_leg.legend(
    handles,
    labels,
    loc="upper left",
    frameon=True,
    fontsize=9.5,
)
radar_path = FIG / "01_deepeval_radar.png"
fig.savefig(radar_path, dpi=160, facecolor="white", pad_inches=0.2)
plt.show()
print("Zapisano:", radar_path)


## 2. Heatmapy DeepEval × typ pytania / kategoria arXiv

Średnie metryk DeepEval w podziale na **typ pytania** (Golden QA) oraz **kategorię arXiv** (grupy z n < 5 → `other`).  
Metryki: **Answer Correctness** i **Faithfulness** (W1–W5).


In [ ]:
def metric_by_question_type(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    return (
        df.pivot_table(index="question_type", columns="variant", values=metric, aggfunc="mean")
        .reindex(columns=[v for v in VARIANT_ORDER if v in df["variant"].unique()])
        .round(4)
    )


def metric_by_category(df: pd.DataFrame, metric: str, min_questions: int = 5) -> pd.DataFrame:
    n_per = df.drop_duplicates("golden_id").groupby("primary_category").size().rename("n_questions")
    tmp = df.copy()
    tmp["category_group"] = tmp["primary_category"].where(
        tmp["primary_category"].map(n_per) >= min_questions, "other"
    )
    pt = (
        tmp.pivot_table(index="category_group", columns="variant", values=metric, aggfunc="mean")
        .reindex(columns=[v for v in VARIANT_ORDER if v in tmp["variant"].unique()])
        .round(4)
    )
    counts = tmp.drop_duplicates("golden_id").groupby("category_group").size().rename("n_questions")
    return pt.join(counts).sort_values("n_questions", ascending=False)


def _draw_matrix_heatmap(
    ax,
    data: pd.DataFrame,
    ylabels: list[str],
    title: str,
    *,
    fontsize: int = 9,
):
    ax.set_facecolor("white")
    im = ax.imshow(data.values, aspect="auto", vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(data.columns)))
    ax.set_xticklabels([LABELS.get(c, c) for c in data.columns], fontsize=11, color="#333333")
    ax.set_yticks(range(len(data.index)))
    ax.set_yticklabels(ylabels, fontsize=10, color="#333333")
    ax.set_title(title, fontsize=13, color="#333333", pad=12)
    ax.tick_params(colors="#444444")
    ax.grid(False)
    nrows, ncols = data.shape
    ax.set_xticks(np.arange(ncols + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(nrows + 1) - 0.5, minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=1.2)
    ax.tick_params(which="minor", bottom=False, left=False)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data.values[i, j]
            if pd.notna(val):
                ax.text(
                    j, i, f"{val:.2f}", ha="center", va="center", fontsize=fontsize,
                    color="#111111" if val < 0.75 else "#ffffff",
                )
    return im


def plot_metric_heatmap_pair(
    by_type: pd.DataFrame,
    by_cat: pd.DataFrame,
    metric_label: str,
    outfile: str,
) -> Path:
    cat_cols = [c for c in by_cat.columns if c != "n_questions"]
    cat_plot = by_cat[cat_cols]
    type_ylabels = [
        QUESTION_TYPE_LABELS_PL.get(str(i), str(i).capitalize()) for i in by_type.index
    ]
    cat_ylabels = [
        f"{idx} (n={int(by_cat.loc[idx, 'n_questions'])})" for idx in cat_plot.index
    ]
    fig_h = max(4.8, 0.42 * len(cat_plot) + 1.5)
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(17.5, fig_h),
        facecolor="white",
        gridspec_kw={"width_ratios": [1.0, 1.25], "wspace": 0.35},
    )
    im = _draw_matrix_heatmap(
        axes[0],
        by_type,
        type_ylabels,
        f"{metric_label} według typu pytania",
    )
    _draw_matrix_heatmap(
        axes[1],
        cat_plot,
        cat_ylabels,
        f"{metric_label} według kategorii arXiv",
        fontsize=8,
    )
    cbar = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.03)
    cbar.set_label(metric_label, color="#444444")
    cbar.ax.tick_params(colors="#444444")
    fig.tight_layout()
    path = FIG / outfile
    fig.savefig(path, dpi=160, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path


saved = []
for idx, metric in enumerate(HEATMAP_METRICS, start=2):
    label = METRIC_COLS_PL[metric]

    by_type = metric_by_question_type(per_q, metric)
    by_type.to_csv(DE / f"{metric}_by_question_type_{BATCH_ID}.csv")
    by_cat = metric_by_category(per_q, metric)
    by_cat.to_csv(DE / f"{metric}_by_primary_category_{BATCH_ID}.csv")

    p = plot_metric_heatmap_pair(
        by_type,
        by_cat,
        label,
        f"{idx:02d}_{metric}_heatmaps.png",
    )
    saved.append(p)

    display(Markdown(f"### {label}"))
    display(by_type.round(3))
    display(by_cat.round(3))

for p in saved:
    print("Zapisano:", p)
